In [1]:
import pandas as pd
import joblib
import numpy as np
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
import requests
import psycopg2

In [4]:
team_codes = {
            1 : "NJD", 2 : "NYI", 3 : "NYR", 4 : "PHI", 5 : "PIT",
            6 : "BOS", 7 : "BUF", 8 : "MTL", 9 : "OTT", 10 : "TOR",
            11: "ATL", 12 : "CAR", 13 : "FLA", 14 : "TBL", 15 : "WSH",
            16 : "CHI", 17 : "DET", 18 : "NSH", 19 : "STL", 20 : "CGY",
            21 : "COL", 22 : "EDM", 23 : "VAN", 24 : "ANA", 25 : "DAL",
            26 : "LAK", 27 : "PHX", 28 : "SJS", 29 : "CBJ", 30 : "MIN",
            31 : "TBL", 32 : "QUE", 33 : "WIN", 34 : "NSH", 35 : "CLR",
            36 : "SEN", 37 : "HAM", 38 : "PIR", 39 : "QUA", 40 : "DCG",
            41 : "MWN", 42 : "QBD", 43 : "MMR", 44 : "NYA", 45 : "SLE",
            46 : "OAK", 47 : "AFM", 48 : "KCS", 49 : "CLE", 50 : "DFL",
            51 : "BRK", 52 : "WPG", 53 : "ARI", 54 : "VGK", 55 : "SEA",
            56 : "CGD", 57 : "TAN", 58: "TSP", 59 : "UTA"
}

team_names = {y: x for x, y in team_codes.items()}

def db_request(query):
  hostname = 'database-1.c52o66wwou5d.us-east-2.rds.amazonaws.com'
  database = 'Sports'
  username = 'postgres'
  pwd = 'CS465Project1-4*'
  port_id = 5432

  conn = None
  crsr = None

  try:
      conn = psycopg2.connect(
          host = hostname,
          dbname = database,
          user = username,
          password = pwd,
          port = port_id
      )

      crsr = conn.cursor()
      crsr.execute(query)
      ret = crsr.fetchall()
      conn.commit()
      return ret
  except Exception as error:
      print(error)
  finally:
      if crsr is not None:
          crsr.close()
      if conn is not None:
          conn.close()

def get_team_data(home_team, away_team, year):
  all_games = []
  home_season_query = f'''Select gamesplayed, xgoalsfor, xgoalsagainst, shotsongoalfor, xshotattmeptspercentage, shotsongoalagainst, pppercentage, pkpercentage
          from teamseasonstats where season_team_id = {year}{year+1}{home_team}
          '''
  away_season_query = f'''Select gamesplayed, xgoalsfor, xgoalsagainst, shotsongoalfor, xshotattmeptspercentage, shotsongoalagainst, pppercentage, pkpercentage
          from teamseasonstats where season_team_id = {year}{year+1}{away_team}
          '''

  h_data = db_request(home_season_query)
  h_data = h_data[0]

  h_gf = int(h_data[0] * h_data[1])
  h_ga = int(h_data[0] * h_data[2])

  a_data = db_request(away_season_query)
  a_data = a_data[0]
  a_gf = int(a_data[0] * a_data[1])
  a_ga = int(a_data[0] * a_data[2])

  row = {
        #"Home_GF": h_gf,
        "Home_GF_Pct": h_gf / (h_gf + h_ga),
        #"Home_GA": h_ga,
        "Home_Shots_For": h_data[3],
        "Home_Shooting_Pct": h_data[4],
        "Home_Save_Pct": h_ga / h_data[5],
        "Home_PDO": h_data[4] + (h_ga / h_data[5]),
        "Home_PP_Pct": h_data[6],
        "Home_PK_Pct": h_data[7],
        #"Away_GF": a_gf,
        "Away_GF_Pct": a_gf / (a_gf + a_ga),
        #"Away_GA": a_ga,
        "Away_Shots_For": a_data[3],
        "Away_Shooting_Pct": a_data[4],
        "Away_Save_Pct": a_ga / a_data[5],
        "Away_PDO": a_data[4] + (a_ga / a_data[5]),
        "Away_PP_Pct": a_data[6],
        "Away_PK_Pct": a_data[7]
  }
  print(row)
  all_games.append(row)
  X = pd.DataFrame(all_games)
  return X

def get_WP(home_team, away_team):

  if type(home_team) != int:
    home_team = team_names[home_team]
    away_team = team_names[away_team]

  # Load the trained model
  model = keras.models.load_model('/content/drive/MyDrive/CS465_Project/Code/Full_NHL_Neural_Net/NHL_Win_Prediction.keras')

  x = get_team_data(home_team,away_team, 2024)

  scaler = joblib.load('/content/drive/MyDrive/CS465_Project/Code/Full_NHL_Neural_Net/scaler.pkl')
  X_new_scaled = scaler.transform(x)

  y_pred_prob = model.predict(X_new_scaled)

  print(float(y_pred_prob[0][0]))
  return float(y_pred_prob[0][0])

get_WP("WSH", "SJS")


{'Home_GF_Pct': 0.5902777777777778, 'Home_Shots_For': 1928, 'Home_Shooting_Pct': 0.4995937318630296, 'Home_Save_Pct': 0.09286463798530954, 'Home_PDO': 0.5924583698483391, 'Home_PP_Pct': 0.188481, 'Home_PK_Pct': 0.857843, 'Away_GF_Pct': 0.4147465437788018, 'Away_Shots_For': 1888, 'Away_Shooting_Pct': 0.46268824454933694, 'Away_Save_Pct': 0.11550704865848113, 'Away_PDO': 0.578195293207818, 'Away_PP_Pct': 0.15736, 'Away_PK_Pct': 0.766355}
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
0.7685201168060303


0.7685201168060303